# GAT HIV 5-Fold Statistics Revize
This notebook runs the GAT model with stratified 5-fold CV and produces `mean ± std`, variance, fold results, and a final table.

In [ ]:


!pip install rdkit

!pip install torch-geometric
!pip install tensorflow

In [ ]:

# Setup (Colab / Jupyter)
import sys, subprocess, pkgutil

def ensure(pkg, install_name=None):
    if pkgutil.find_loader(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", install_name or pkg])

ensure("rdkit", "rdkit-pypi")
ensure("torch_geometric", "torch-geometric")

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from rdkit import Chem
from rdkit.Chem import rdmolops, HybridizationType

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:

SEED = 42
BATCH_SIZE = 32
LR = 3e-4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 120
PATIENCE = 12

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def atom_features(atom):
    return [
        atom.GetAtomicNum(),
        atom.GetTotalDegree(),
        atom.GetFormalCharge(),
        int(atom.GetHybridization() == HybridizationType.SP),
        int(atom.GetHybridization() == HybridizationType.SP2),
        int(atom.GetHybridization() == HybridizationType.SP3),
        int(atom.GetIsAromatic()),
        int(atom.IsInRing())
    ]

def mol_to_graph(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0:
        return None
    adj = rdmolops.GetAdjacencyMatrix(mol)
    feats = [atom_features(atom) for atom in mol.GetAtoms()]
    x = torch.tensor(feats, dtype=torch.float)
    edge_index = torch.tensor(np.array(np.nonzero(adj)), dtype=torch.long)
    y = torch.tensor([float(label)], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, y=y)

df = pd.read_csv("https://raw.githubusercontent.com/McahitKutsal/hivcsv/main/HIV7.csv")
df = df.dropna(subset=["smiles", "HIV_active"]).reset_index(drop=True)

graphs = []
labels = []
for _, row in df.iterrows():
    g = mol_to_graph(row["smiles"], row["HIV_active"])
    if g is not None:
        graphs.append(g)
        labels.append(int(row["HIV_active"]))

labels = np.array(labels)
print("Total valid graphs:", len(graphs))
print("Positive ratio:", labels.mean())


In [4]:

class GATModel(nn.Module):
    def __init__(self, in_channels=8):
        super().__init__()
        self.gat1 = GATConv(in_channels, 128, heads=4, concat=False, dropout=0.3)
        self.bn1  = nn.BatchNorm1d(128)
        self.gat2 = GATConv(128, 64, heads=4, concat=False, dropout=0.3)
        self.bn2  = nn.BatchNorm1d(64)
        self.dropout = nn.Dropout(0.4)
        self.lin1 = nn.Linear(64, 32)
        self.lin2 = nn.Linear(32, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.elu(self.gat1(x, edge_index))
        x = self.bn1(x)
        x = F.elu(self.gat2(x, edge_index))
        x = self.bn2(x)
        x = global_mean_pool(x, batch)
        x = self.dropout(x)
        x = F.relu(self.lin1(x))
        return torch.sigmoid(self.lin2(x)).view(-1)

def evaluate_model(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    preds, labels = [], []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data)
            loss = loss_fn(out, data.y.view(-1).float())
            total_loss += loss.item() * data.num_graphs
            preds.extend(out.detach().cpu().numpy().tolist())
            labels.extend(data.y.view(-1).cpu().numpy().tolist())
    avg_loss = total_loss / len(loader.dataset)
    pred_binary = [1 if p >= 0.5 else 0 for p in preds]
    acc = accuracy_score(labels, pred_binary)
    precision = precision_score(labels, pred_binary, zero_division=0)
    recall = recall_score(labels, pred_binary, zero_division=0)
    f1 = f1_score(labels, pred_binary, zero_division=0)
    auc = roc_auc_score(labels, preds) if len(set(labels)) > 1 else 0.0
    return avg_loss, acc, precision, recall, f1, auc, preds, labels

def train_one_fold(train_graphs, val_graphs, test_graphs, fold_id):
    train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_graphs, batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(test_graphs, batch_size=BATCH_SIZE, shuffle=False)

    model = GATModel().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCELoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5
    )

    best_val_loss = float("inf")
    best_state_dict = None
    wait = 0

    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    train_aucs, val_aucs = [], []

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        train_preds, train_labels = [], []

        for data in train_loader:
            data = data.to(device)
            optimizer.zero_grad()
            out = model(data)
            loss = loss_fn(out, data.y.view(-1).float())
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * data.num_graphs
            train_preds.extend(out.detach().cpu().numpy().tolist())
            train_labels.extend(data.y.view(-1).cpu().numpy().tolist())

        avg_train_loss = total_loss / len(train_loader.dataset)
        train_pred_binary = [1 if p >= 0.5 else 0 for p in train_preds]
        train_acc = accuracy_score(train_labels, train_pred_binary)
        train_auc = roc_auc_score(train_labels, train_preds) if len(set(train_labels)) > 1 else 0.0

        val_loss, val_acc, _, _, _, val_auc, _, _ = evaluate_model(model, val_loader, loss_fn)
        scheduler.step(val_loss)

        train_losses.append(avg_train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)
        train_aucs.append(train_auc)
        val_aucs.append(val_auc)

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f"Fold {fold_id}: Early stopping at epoch {epoch}")
                break

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    test_loss, test_acc, test_precision, test_recall, test_f1, test_auc, _, _ = evaluate_model(model, test_loader, loss_fn)
    history = {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "train_accs": train_accs,
        "val_accs": val_accs,
        "train_aucs": train_aucs,
        "val_aucs": val_aucs,
    }
    metrics = {
        "fold": fold_id,
        "test_accuracy": test_acc,
        "test_precision": test_precision,
        "test_recall": test_recall,
        "test_f1": test_f1,
        "test_roc_auc": test_auc,
        "epochs_ran": len(train_losses),
    }
    return metrics, history


In [ ]:

all_fold_results = []
all_histories = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
indices = np.arange(len(graphs))

for fold_id, (train_idx, temp_idx) in enumerate(skf.split(indices, labels), start=1):
    train_idx = np.array(train_idx)
    temp_idx = np.array(temp_idx)

    temp_labels = labels[temp_idx]
    val_sub_idx, test_sub_idx = train_test_split(
        np.arange(len(temp_idx)),
        test_size=0.5,
        random_state=SEED + fold_id,
        stratify=temp_labels
    )

    train_graphs = [graphs[i] for i in train_idx]
    val_graphs = [graphs[temp_idx[i]] for i in val_sub_idx]
    test_graphs = [graphs[temp_idx[i]] for i in test_sub_idx]

    print(f"===== Fold {fold_id} =====")
    print("Train:", len(train_graphs), "Val:", len(val_graphs), "Test:", len(test_graphs))

    fold_result, history = train_one_fold(train_graphs, val_graphs, test_graphs, fold_id)
    print(fold_result)

    all_fold_results.append(fold_result)
    all_histories.append(history)

results_df = pd.DataFrame(all_fold_results)
results_df


In [ ]:

results_df.to_csv("GAT_fold_results.csv", index=False)

summary_rows = []
for metric in ["test_accuracy", "test_precision", "test_recall", "test_f1", "test_roc_auc"]:
    mean_val = results_df[metric].mean()
    std_val = results_df[metric].std(ddof=1)
    var_val = results_df[metric].var(ddof=1)
    summary_rows.append({
        "Metric": metric.replace("test_", "").upper(),
        "Mean": mean_val,
        "Std": std_val,
        "Variance": var_val,
        "Formatted": f"{mean_val:.3f} ± {std_val:.3f}",
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("GAT_summary_results.csv", index=False)
summary_df


In [ ]:

final_table = pd.DataFrame([{
    "Model": "GAT",
    "Accuracy": summary_df.loc[summary_df["Metric"] == "ACCURACY", "Formatted"].iloc[0],
    "Precision": summary_df.loc[summary_df["Metric"] == "PRECISION", "Formatted"].iloc[0],
    "Recall": summary_df.loc[summary_df["Metric"] == "RECALL", "Formatted"].iloc[0],
    "F1": summary_df.loc[summary_df["Metric"] == "F1", "Formatted"].iloc[0],
    "ROC-AUC": summary_df.loc[summary_df["Metric"] == "ROC_AUC", "Formatted"].iloc[0],
}])

final_table.to_csv("GAT_final_table.csv", index=False)
final_table


In [ ]:

# Mean training plots
def pad_and_average(series_list):
    max_len = max(len(x) for x in series_list)
    arr = np.full((len(series_list), max_len), np.nan, dtype=float)
    for i, s in enumerate(series_list):
        arr[i, :len(s)] = s
    return np.nanmean(arr, axis=0)

avg_train_acc = pad_and_average([h["train_accs"] for h in all_histories])
avg_val_acc   = pad_and_average([h["val_accs"] for h in all_histories])
avg_train_loss = pad_and_average([h["train_losses"] for h in all_histories])
avg_val_loss   = pad_and_average([h["val_losses"] for h in all_histories])

epochs = np.arange(1, len(avg_train_acc) + 1)

plt.figure(figsize=(8,4))
plt.plot(epochs, avg_train_acc, label="Train")
plt.plot(epochs, avg_val_acc, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("GAT Average Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8,4))
plt.plot(epochs, avg_train_loss, label="Train")
plt.plot(epochs, avg_val_loss, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("GAT Average Loss")
plt.legend()
plt.grid(True)
plt.show()


## GAT docking preparation block

This block was added to be run after the current training is completed.

Goal:
- build a 13-column table from existing prediction outputs
- select the top 10 candidates
- determine the final 2 candidates
- colored 2D drawing yapmak
- `.smi` docking file

In [ ]:
# Colab RDKit install
import sys, subprocess, pkgutil

if pkgutil.find_loader("rdkit") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdkit-pypi"])

from rdkit import Chem
from rdkit.Chem import Descriptors, Draw
from rdkit.Chem import Lipinski as RdLipinski, Crippen, QED

print("RDKit OK")

In [ ]:
# ==========================================
# GAT FINAL PIPELINE (SELF-CONTAINED, ROBUST)
# ==========================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch_geometric.loader import DataLoader

from rdkit import Chem
from rdkit.Chem import Descriptors, Draw
from rdkit.Chem import Lipinski as RdLipinski, Crippen, QED

# 1) Select the best fold
auc_col = "test_roc_auc" if "test_roc_auc" in results_df.columns else results_df.columns[-1]
best_fold_idx = int(results_df[auc_col].astype(float).idxmax())
best_fold_number = best_fold_idx + 1

print(f"Using AUC column: {auc_col}")
print(f"Using best fold: {best_fold_number}")

# 2) Rebuild the same split
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
indices = np.arange(len(graphs))
splits = list(skf.split(indices, labels))

train_idx, temp_idx = splits[best_fold_idx]
train_idx = np.array(train_idx)
temp_idx = np.array(temp_idx)

temp_labels = np.array(labels)[temp_idx]
val_sub_idx, test_sub_idx = train_test_split(
    np.arange(len(temp_idx)),
    test_size=0.5,
    random_state=SEED + best_fold_number,
    stratify=temp_labels
)

train_graphs = [graphs[i] for i in train_idx]
val_graphs   = [graphs[temp_idx[i]] for i in val_sub_idx]
test_graphs  = [graphs[temp_idx[i]] for i in test_sub_idx]

test_global_idx = temp_idx[test_sub_idx]
smiles_test = df.iloc[test_global_idx]["smiles"].reset_index(drop=True)
y_test = df.iloc[test_global_idx]["HIV_active"].astype(float).values

print("Train:", len(train_graphs), "Val:", len(val_graphs), "Test:", len(test_graphs))

# 3) DataLoader
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_graphs, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_graphs, batch_size=BATCH_SIZE, shuffle=False)

# 4) Rebuild and train the model
set_seed(SEED + best_fold_number)

model = GATModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
loss_fn = nn.BCELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5
)

best_val_loss = float("inf")
best_state_dict = None
wait = 0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss = 0.0

    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data)
        loss = loss_fn(out, data.y.view(-1).float())
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs

    # validation
    model.eval()
    val_total_loss = 0.0
    with torch.no_grad():
        for data in val_loader:
            data = data.to(device)
            out = model(data)
            loss = loss_fn(out, data.y.view(-1).float())
            val_total_loss += loss.item() * data.num_graphs

    val_loss = val_total_loss / len(val_loader.dataset)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

# restore the best model
model.load_state_dict(best_state_dict)
model = model.to(device)
model.eval()

# 5) Test prediction
test_preds = []
with torch.no_grad():
    for data in test_loader:
        data = data.to(device)
        out = model(data)
        test_preds.extend(out.detach().cpu().numpy().tolist())

y_prob = np.array(test_preds).reshape(-1)
y_pred = (y_prob >= 0.5).astype(int)

# 6) Prediction dataframe
df_pred = pd.DataFrame({
    "fold": best_fold_number,
    "smiles": smiles_test,
    "y_true": y_test,
    "y_pred": y_pred,
    "y_prob": y_prob
})

# 7) Top 10 candidates
top_df = df_pred.sort_values(by="y_prob", ascending=False).head(10).reset_index(drop=True)

# 8) Descriptor hesaplama
def compute_desc(sm):
    mol = Chem.MolFromSmiles(sm)
    if mol is None:
        return None

    mw = Descriptors.MolWt(mol)
    logp = Crippen.MolLogP(mol)
    hbd = RdLipinski.NumHDonors(mol)
    hba = RdLipinski.NumHAcceptors(mol)
    tpsa = Descriptors.TPSA(mol)
    rot = RdLipinski.NumRotatableBonds(mol)
    qed = QED.qed(mol)
    lip = (mw <= 500) and (logp <= 5) and (hbd <= 5) and (hba <= 10)

    return {
        "MW": mw,
        "LogP": logp,
        "HBD": hbd,
        "HBA": hba,
        "TPSA": tpsa,
        "RotatableBonds": rot,
        "QED": qed,
        "Lipinski": lip
    }

rows = []
for _, row in top_df.iterrows():
    d = compute_desc(row["smiles"])
    if d is not None:
        d.update(row.to_dict())
        rows.append(d)

df_desc = pd.DataFrame(rows)

# 9) Shared 13 columns
df_desc = df_desc[[
    "MW", "LogP", "HBD", "HBA", "TPSA", "RotatableBonds",
    "QED", "smiles", "fold", "y_true", "y_pred", "y_prob", "Lipinski"
]]

print("\nGAT TOP 10 CANDIDATES:")
display(df_desc)

# 10) Final 2 candidates
filtered_df = df_desc[df_desc["Lipinski"] == True].copy()

if len(filtered_df) >= 2:
    final_df = filtered_df.sort_values(by=["QED", "y_prob"], ascending=False).head(2).copy()
else:
    final_df = df_desc.sort_values(by=["QED", "y_prob"], ascending=False).head(2).copy()

final_df = final_df[[
    "MW", "LogP", "HBD", "HBA", "TPSA", "RotatableBonds",
    "QED", "smiles", "fold", "y_true", "y_pred", "y_prob", "Lipinski"
]]

print("\nGAT FINAL 2 CANDIDATES:")
display(final_df)

# 11) Save
df_desc.to_csv("GAT_top_10_candidates.csv", index=False)
final_df.to_csv("GAT_final_2_candidates.csv", index=False)
final_df["smiles"].to_csv("GAT_docking_input.smi", index=False, header=False)

print("\nSaved: GAT_top_10_candidates.csv")
print("Saved: GAT_final_2_candidates.csv")
print("Saved: GAT_docking_input.smi")

# 12) Colored drawing
mols = [Chem.MolFromSmiles(sm) for sm in final_df["smiles"]]

img = Draw.MolsToGridImage(
    mols,
    molsPerRow=2,
    subImgSize=(340, 340),
    legends=[
        f"GAT Mol1\nProb={final_df.iloc[0]['y_prob']:.3f}" if len(final_df) > 0 else "",
        f"GAT Mol2\nProb={final_df.iloc[1]['y_prob']:.3f}" if len(final_df) > 1 else ""
    ]
)

display(img)